# Chapter 5 Lab — Language Models: N-grams to GPT

Three models on the same small public corpus (NLTK Reuters sample): a smoothed bigram/trigram
model, a from-scratch character-level LSTM, and pretrained GPT-2 generation for comparison.

In [ ]:
import nltk
try:
    nltk.download("reuters", quiet=True)
    nltk.download("punkt", quiet=True)
    from nltk.corpus import reuters
    text = " ".join(reuters.words()[:20000]).lower()
    corpus_mode = "NLTK Reuters"
except Exception as exc:
    text = ("the market opened higher after the company reported earnings. "
            "the cat sat on the mat and watched the market news. " * 250).lower()
    corpus_mode = f"synthetic fallback ({type(exc).__name__})"
print("Corpus mode:", corpus_mode)
print(text[:200])


## 1. Bigram/trigram model with add-k smoothing

In [ ]:
from collections import defaultdict, Counter
import re, random

tokens = re.findall(r"[a-z']+", text)

def train_ngram(tokens, n=2):
    model = defaultdict(Counter)
    for i in range(len(tokens) - n + 1):
        ctx, nxt = tuple(tokens[i:i+n-1]), tokens[i+n-1]
        model[ctx][nxt] += 1
    return model

bigram_model = train_ngram(tokens, n=2)

def generate(model, start, n=2, length=20, k=1.0):
    ctx = tuple(start.split())
    out = list(ctx)
    for _ in range(length):
        counts = model.get(ctx, Counter())
        vocab = list(counts.keys()) or ["the"]
        weights = [counts.get(w, 0) + k for w in vocab]
        nxt = random.choices(vocab, weights=weights)[0]
        out.append(nxt)
        ctx = tuple(out[-(n-1):])
    return " ".join(out)

print(generate(bigram_model, "the", n=2, length=15))

## 2. Perplexity

In [ ]:
import math

def perplexity(model, tokens, n=2, k=1.0, vocab_size=5000):
    logp = 0
    for i in range(len(tokens) - n + 1):
        ctx, nxt = tuple(tokens[i:i+n-1]), tokens[i+n-1]
        counts = model.get(ctx, Counter())
        total = sum(counts.values()) + k * vocab_size
        p = (counts.get(nxt, 0) + k) / total
        logp += math.log(p)
    return math.exp(-logp / (len(tokens) - n + 1))

print("Perplexity:", perplexity(bigram_model, tokens[:2000]))

## 3. A tiny character-level LSTM (from scratch, short training loop)

In [ ]:
import torch, torch.nn as nn

chars = sorted(set(text[:5000]))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text[:5000]])

class CharLSTM(nn.Module):
    def __init__(self, vocab_size, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, hidden)
        self.lstm = nn.LSTM(hidden, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)
    def forward(self, x, hc=None):
        e = self.emb(x)
        out, hc = self.lstm(e, hc)
        return self.fc(out), hc

model = CharLSTM(len(chars))
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
seq_len = 32
for step in range(200):
    i = random.randint(0, len(data) - seq_len - 1)
    x = data[i:i+seq_len].unsqueeze(0)
    y = data[i+1:i+seq_len+1].unsqueeze(0)
    logits, _ = model(x)
    loss = nn.functional.cross_entropy(logits.view(-1, len(chars)), y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
print("final training loss:", loss.item())

## 4. Pretrained GPT-2 generation (the bridge to Part III)

In [ ]:
try:
    from transformers import pipeline
    gen = pipeline("text-generation", model="gpt2", local_files_only=True)
    print(gen("The cat sat on the", max_new_tokens=15, num_return_sequences=1)[0]["generated_text"])
except Exception as e:
    print("transformers/gpt2 unavailable offline ? fallback: The cat sat on the mat and watched the room.", type(e).__name__)


## Exercise

Compare the fluency of the bigram generator, the tiny char-LSTM (undertrained on purpose, to
keep this notebook fast), and GPT-2. Which jump in quality is bigger: bigram→LSTM or LSTM→GPT-2?
What does that suggest about the relative contribution of architecture vs. scale?